# บท 09 · วงจรคำสั่งกับโบรกเกอร์จำลอง

ใช้ USD และหุ้น DEMO ที่สร้างเอง ไม่เชื่อมบัญชี ไม่มี key และไม่มี network `PaperBroker` เป็นคลาสของหลักสูตร ไม่ใช่ Webull SDK รันจาก kernel ใหม่ตามลำดับ

## 1. กำหนดสถานะคำสั่งและหน่วยเงิน

ใช้ Decimal สำหรับราคาฐานสิบ จำนวนหุ้นเป็นจำนวนเต็ม สถานะเป็นชื่อภายใน mock ไม่ใช่รหัสรับรองจาก Webull

In [1]:
"""Offline broker contract demonstration; these names are NOT Webull SDK methods."""
from dataclasses import dataclass
from decimal import Decimal
import json

D = Decimal

@dataclass
class MockOrder:
    intent_id: str
    symbol: str
    quantity: int
    limit: Decimal
    filled: int = 0
    average: Decimal = D("0")
    status: str = "ACCEPTED"


print(MockOrder("example", "DEMO", 10, D("100.00")))


MockOrder(intent_id='example', symbol='DEMO', quantity=10, limit=Decimal('100.00'), filled=0, average=Decimal('0'), status='ACCEPTED')


## 2. โบรกเกอร์จำลอง

รับ intent เดิมแล้วไม่สร้างคำสั่งเพิ่ม ตรวจ fill ID ซ้ำและ payload ขัดกัน แบบฝึกหัดจำกัดเป็น BUY-only ไม่จำลอง settlement, margin, cancel/fill race หรือการเก็บ state ข้าม process

In [2]:
class PaperBroker:
    """Deliberately small BUY-only simulator with deterministic partial fills."""
    def __init__(self, cash="2000.00"):
        self.cash = D(cash)
        self.position = 0
        self.orders = {}
        self.fill_ids = {}
        self.fees = D("0")

    def submit(self, intent_id, symbol, quantity, limit, lose_response=False):
        limit = D(limit)
        if intent_id in self.orders:
            old = self.orders[intent_id]
            if (old.symbol, old.quantity, old.limit) != (symbol, quantity, limit):
                raise ValueError("Intent ID reused with a different payload")
            return old
        if type(quantity) is not int or quantity <= 0 or limit <= 0:
            raise ValueError("Positive integer quantity and positive limit required")
        reserved = sum((o.quantity - o.filled) * o.limit for o in self.orders.values()
                       if o.status in ("ACCEPTED", "PARTIAL"))
        if D(quantity) * limit > self.cash - reserved:
            raise ValueError("Insufficient unreserved cash")
        order = MockOrder(intent_id, symbol, quantity, limit)
        self.orders[intent_id] = order
        if lose_response:
            raise TimeoutError("Mock accepted order but its response was lost")
        return order

    def query(self, intent_id):
        return self.orders.get(intent_id)

    def apply_fill(self, intent_id, fill_id, quantity, price, fee="0.00"):
        price, fee = D(price), D(fee)
        signature = (intent_id, quantity, price, fee)
        if fill_id in self.fill_ids:
            if self.fill_ids[fill_id] != signature:
                raise ValueError("Fill ID reused with a different payload")
            return "DUPLICATE_IGNORED"
        order = self.orders[intent_id]
        if order.status not in ("ACCEPTED", "PARTIAL"):
            raise ValueError("Order is not open")
        if type(quantity) is not int or quantity <= 0 or quantity > order.quantity - order.filled:
            raise ValueError("Invalid fill quantity")
        if not D("0") < price <= order.limit or fee < 0:
            raise ValueError("Invalid fill price or fee")
        debit = price * quantity + fee
        if debit > self.cash:
            raise ValueError("Insufficient cash including fees")
        order.average = (order.average * order.filled + price * quantity) / (order.filled + quantity)
        order.filled += quantity
        order.status = "FILLED" if order.filled == order.quantity else "PARTIAL"
        self.cash -= debit
        self.position += quantity
        self.fees += fee
        self.fill_ids[fill_id] = signature
        return order.status

    def cancel(self, intent_id):
        order = self.orders[intent_id]
        if order.status in ("ACCEPTED", "PARTIAL"):
            order.status = "CANCELED"
        return order.status


print("PaperBroker ready: offline simulation only")


PaperBroker ready: offline simulation only


## 3. Timeout หลังรับคำสั่ง และ partial fills

ซื้อ 10 หุ้นที่ limit 100: fill 4 หุ้นที่ 99.90 + fee 0.40 แล้ว 6 หุ้นที่ 100 + fee 0.60 รวมเงินใช้ 1,000.60 USD ค่าจับคู่เป็นสมมติฐาน ไม่ใช่ผลตลาด

In [3]:
def remaining_target(target, broker):
    pending = sum(o.quantity - o.filled for o in broker.orders.values()
                  if o.status in ("ACCEPTED", "PARTIAL"))
    return target - broker.position - pending

def demonstration():
    broker = PaperBroker()
    key = "DEMO|2026-01-05T14:31:00Z|v1|BUY10"
    try:
        broker.submit(key, "DEMO", 10, "100.00", lose_response=True)
    except TimeoutError:
        recovered = broker.query(key)
        assert recovered is not None
    statuses = [broker.query(key).status]
    statuses.append(broker.apply_fill(key, "fill-1", 4, "99.90", "0.40"))
    after_partial = dict(cash=str(broker.cash), position=broker.position,
                         pending=broker.query(key).quantity-broker.query(key).filled,
                         additional_order=remaining_target(10, broker))
    duplicate = broker.apply_fill(key, "fill-1", 4, "99.90", "0.40")
    statuses.append(broker.apply_fill(key, "fill-2", 6, "100.00", "0.60"))
    return broker, key, dict(statuses=statuses, partial=after_partial, duplicate=duplicate,
        cash=str(broker.cash), position=broker.position,
        average_price=str(broker.query(key).average), fees=str(broker.fees),
        equity_at_100=str(broker.cash + broker.position*D("100")))


broker, key, result = demonstration()
print(json.dumps(result, indent=2))


{
  "statuses": [
    "ACCEPTED",
    "PARTIAL",
    "FILLED"
  ],
  "partial": {
    "cash": "1600.00",
    "position": 4,
    "pending": 6,
    "additional_order": 0
  },
  "duplicate": "DUPLICATE_IGNORED",
  "cash": "999.40",
  "position": 10,
  "average_price": "99.96",
  "fees": "1.00",
  "equity_at_100": "1999.40"
}


## 4. ทำไม 10 − 4 ยังไม่ใช่จำนวนที่ต้องส่งใหม่

ต้องลบ pending buy ด้วย เป้าหมาย 10 ถือ 4 และค้างอีก 6 จึงยังไม่ต้องส่งคำสั่งเพิ่ม

In [4]:
partial = PaperBroker()
partial.submit("partial-demo", "DEMO", 10, "100")
partial.apply_fill("partial-demo", "p1", 4, "99.90", "0.40")
print("held:", partial.position)
print("additional order:", remaining_target(10, partial))
assert remaining_target(10, partial) == 0
print("cash:", partial.cash)


held: 4
additional order: 0
cash: 1600.00


## 5. แบบฝึกหัด: ยกเลิกส่วนที่เหลือ

mock ยืนยัน cancel ทันที หุ้นที่ fill แล้วไม่หาย หลังซื้อ 4 หุ้นที่ 100 เสีย fee 0.40 เหลือเงิน 1,599.60 USD หากเป้ายังเป็น 10 จะต้องการอีก 6 หุ้น แต่ต้องตัดสินใจใหม่และผ่าน risk gate

In [5]:
canceled = PaperBroker()
canceled.submit("cancel-demo", "DEMO", 10, "100")
canceled.apply_fill("cancel-demo", "c1", 4, "100", "0.40")
print("status:", canceled.cancel("cancel-demo"))
print("held:", canceled.position, "cash:", canceled.cash)
print("additional target:", remaining_target(10, canceled))
assert canceled.cash == D("1599.60")
assert remaining_target(10, canceled) == 6


status: CANCELED
held: 4 cash: 1599.60
additional target: 6


## 6. ตรวจการบัญชีและคำสั่งซ้ำ

การไม่มีคำสั่งซ้ำในหน่วยความจำไม่ได้แก้ restart บท 10 จะใช้ journal และ reconciliation ต่อจากจุดนี้

In [6]:
def checks():
    broker, key, result = demonstration()
    assert result["statuses"] == ["ACCEPTED", "PARTIAL", "FILLED"]
    assert broker.position == 10 and broker.cash == D("999.40")
    assert broker.query(key).average == D("99.96")
    assert remaining_target(10, broker) == 0
    broker.submit(key, "DEMO", 10, "100.00")
    assert len(broker.orders) == 1
    try:
        broker.submit(key, "DEMO", 11, "100.00")
    except ValueError:
        pass
    else:
        raise AssertionError("Payload mismatch must be rejected")
    canceled = PaperBroker()
    canceled.submit("c", "DEMO", 10, "100")
    canceled.apply_fill("c", "c1", 4, "100", "0.4")
    canceled.cancel("c")
    assert canceled.position == 4 and remaining_target(10, canceled) == 6
    try:
        canceled.apply_fill("c", "c2", 6, "100")
    except ValueError:
        pass
    else:
        raise AssertionError("Canceled mock order must reject new fills")
    return "8 broker-state checks passed"


print(checks())
try:
    broker.apply_fill(key, "fill-1", 5, "99.90", "0.40")
except ValueError as exc:
    print("conflicting duplicate rejected:", exc)
else:
    raise AssertionError("Conflicting duplicate was accepted")


8 broker-state checks passed
conflicting duplicate rejected: Fill ID reused with a different payload


## แหล่งอ้างอิงและขอบเขต

[Webull Order Query](https://developer.webull.co.th/apis/docs/reference/trade-api/order-query/), [Assets](https://developer.webull.co.th/apis/docs/reference/trade-api/assets/) และ [Trade Events](https://developer.webull.co.th/apis/docs/reference/custom/trading-events/) ตรวจ 11 กันยายน 2026

Hilpisch บท 8 หน้า 223,229 และบท 9 หน้า 249,256 (PDF 243,249,269,276) เป็นกรอบทำงานกับโบรกเกอร์ Oanda/FXCM ไม่ใช่เอกสาร Webull โค้ดนี้เขียนใหม่ทั้งหมด การใช้งาน API จริงต้องตรวจบัญชี ภูมิภาค สิทธิ์ และ SDK แยกจาก mock